# Kredi Kartı Kümeleme

Bu projede kart kullanımına göre müşterileri gruplayacağım. Pazarlama için segment çıkarmak istiyorum.


In [ ]:
import pandas as pd
pd.set_option('display.max_columns',100)
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/CC_GENERAL.csv')
df.head()


### EDA


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


### Görselleştirme


In [ ]:
sns.histplot(df['BALANCE'],bins=30)
plt.show()


In [ ]:
sns.scatterplot(x='PURCHASES',y='CREDIT_LIMIT',data=df)
plt.show()


### Boş veri


In [ ]:
df['CREDIT_LIMIT']=df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median())
df['MINIMUM_PAYMENTS']=df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].median())


### Feature Engineering


In [ ]:
cols=['BALANCE','PURCHASES','CASH_ADVANCE','CREDIT_LIMIT','PAYMENTS','MINIMUM_PAYMENTS','PRC_FULL_PAYMENT']
x=df[cols]
from sklearn.preprocessing import StandardScaler
x=StandardScaler().fit_transform(x)


### Train Test Split

Kümelemede asıl iş tüm veri ama hocanın istediği gibi böldüm.


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test=train_test_split(x,test_size=0.2,random_state=42)


### 3 Model


In [ ]:
from sklearn.cluster import KMeans,AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

km=KMeans(n_clusters=4,random_state=42,n_init=10)
agg=AgglomerativeClustering(n_clusters=4)
gmm=GaussianMixture(n_components=4,random_state=42)

for ad,lab in [('KMeans',km.fit_predict(x)),('Agglo',agg.fit_predict(x)),('GMM',gmm.fit_predict(x))]:
    print(ad,round(silhouette_score(x,lab),3))


In [ ]:
df['kume']=km.fit_predict(x)
df.groupby('kume')[cols].mean()


In [ ]:
import joblib
joblib.dump(km,'../../models/clustering_credit_card.joblib')


### Sonuç

4 küme mantıklı geldi: az harcayan, nakit avansçı, yüksek limitli vs. KMeans'de silhouette daha iyiydi. Hedefi tutturdum.
